# Assignment 4
## Description
In this assignment you must read in a file of metropolitan regions and associated sports teams from [assets/wikipedia_data.html](assets/wikipedia_data.html) and answer some questions about each metropolitan region. Each of these regions may have one or more teams from the "Big 4": NFL (football, in [assets/nfl.csv](assets/nfl.csv)), MLB (baseball, in [assets/mlb.csv](assets/mlb.csv)), NBA (basketball, in [assets/nba.csv](assets/nba.csv) or NHL (hockey, in [assets/nhl.csv](assets/nhl.csv)). Please keep in mind that all questions are from the perspective of the metropolitan region, and that this file is the "source of authority" for the location of a given sports team. Thus teams which are commonly known by a different area (e.g. "Oakland Raiders") need to be mapped into the metropolitan region given (e.g. San Francisco Bay Area). This will require some human data understanding outside of the data you've been given (e.g. you will have to hand-code some names, and might need to google to find out where teams are)!

For each sport I would like you to answer the question: **what is the win/loss ratio's correlation with the population of the city it is in?** Win/Loss ratio refers to the number of wins over the number of wins plus the number of losses. Remember that to calculate the correlation with [`pearsonr`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html), so you are going to send in two ordered lists of values, the populations from the wikipedia_data.html file and the win/loss ratio for a given sport in the same order. Average the win/loss ratios for those cities which have multiple teams of a single sport. Each sport is worth an equal amount in this assignment (20%\*4=80%) of the grade for this assignment. You should only use data **from year 2018** for your analysis -- this is important!

## Notes

1. Do not include data about the MLS or CFL in any of the work you are doing, we're only interested in the Big 4 in this assignment.
2. I highly suggest that you first tackle the four correlation questions in order, as they are all similar and worth the majority of grades for this assignment. This is by design!
3. It's fair game to talk with peers about high level strategy as well as the relationship between metropolitan areas and sports teams. However, do not post code solving aspects of the assignment (including such as dictionaries mapping areas to teams, or regexes which will clean up names).
4. There may be more teams than the assert statements test, remember to collapse multiple teams in one city into a single value!

As this assignment utilizes global variables in the skeleton code, to avoid having errors in your code you can either:

1. You can place all of your code within the function definitions for all of the questions (other than import statements).
2. You can create copies of all the global variables with the copy() method and proceed as usual.

## Question 1
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NHL** using **2018** data.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

nhl_df=pd.read_csv("assets/nhl.csv")
cities=pd.read_html("assets/wikipedia_data.html")[1]
cities=cities.iloc[:-1,[0,3,5,6,7,8]]

nhl_df_local=nhl_df
cities_local=cities

pattern = r'\[note\s?[0-9]*\]'
for x in cities_local.columns: 
    cities_local[x] = cities_local[x].str.replace(pat = pattern,repl = '', case=False, flags = re.IGNORECASE, regex = True)

pattern = r'[*+]'
for x in cities_local.columns: 
    cities_local[x] = cities_local[x].str.replace(pat = pattern,repl = '', case=False, flags = re.IGNORECASE, regex = True)

cities_local.rename({"Population (2016 est.)[8]": "Population"}, axis =1, inplace = True)
#cities_local.columns = [ x for x in cities_local.columns]
population_by_region = [] # pass in metropolitan area population from cities_local
win_loss_by_region = [] # pass in win/loss ratio from nhl_df_local in the same order as cities_local["Metropolitan area"]

cities_local = cities_local[['Metropolitan area', 'NHL', 'Population']]
cities_local.sort_values(by = 'Metropolitan area', inplace = True)

citites = cities_local.set_index('NHL', inplace = True)
cities_local.drop(labels = ['—',  ''], axis = 0, inplace = True)
cities_local.head(3)
population = cities_local[['Population', 'Metropolitan area']]
population.set_index('Metropolitan area', inplace = True)
population.head(3)
population_by_region = cities_local['Population'].values.astype(int)

nhl_df_local_2018 = nhl_df_local[nhl_df_local['year'] ==2018]
teams_to_drop = ['Atlantic Division', 'Metropolitan Division', 'Central Division', 'Pacific Division'] 

nhl_df_local_2018 = nhl_df_local_2018[~nhl_df_local_2018['team'].isin(teams_to_drop)] 

#nhl_df_local_2018.head()

pattern = r'[*+]'
for x in nhl_df_local_2018.columns: 
    nhl_df_local_2018[x]=nhl_df_local_2018[x].astype('str')
    nhl_df_local_2018[x] = nhl_df_local_2018[x].str.replace(pat = pattern,repl = '', case=False, flags = re.IGNORECASE, regex = True)

nhl_df_local_2018['area'] = nhl_df_local_2018['team'].apply(lambda x: x.split(' ')[-1])
#print(nhl_df_local_2018.head(5))
#print(cities_local.index.values)
#print(nhl_df_local_2018[['team','area']])

def lookup_area_by_team(nfl_team):
    for team in cities_local.index.values:
        if nfl_team in team:
            return cities_local.at[team,'Metropolitan area']

nhl_df_local_2018['area'] = nhl_df_local_2018['area'].apply(lookup_area_by_team)

from IPython.display import HTML

#print(nhl_df_local_2018['team'].apply(lambda x: x.split(' ')[-1])
display(HTML(nhl_df_local_2018.to_html()))

nhl_df_local_2018['W']=nhl_df_local_2018['W'].astype('int')
nhl_df_local_2018['L']=nhl_df_local_2018['L'].astype('int')
nhl_df_local_2018['win_loss_ratio'] = nhl_df_local_2018['W']/ (nhl_df_local_2018['W'] + nhl_df_local_2018['L'])
nhl_df_local_2018_grouped = nhl_df_local_2018.groupby(by= ['area']).agg(win = ('W','sum') , loss=( 'L', 'sum'))
nhl_df_local_2018_grouped['win_loss_ratio']  = nhl_df_local_2018_grouped['win'] /  (nhl_df_local_2018_grouped['win']+ nhl_df_local_2018_grouped['loss'] )
nhl_df_local_2018_grouped 
#nhl_df_local_2018_grouped['win_loss_ratio'] =nhl_df_local_2018_grouped['win']/(nhl_df_local_2018_grouped['win']+ nhl_df_local_2018_grouped['loss'])

#print(population)
display(HTML(nhl_df_local_2018_grouped.to_html()))

out_df_nhl = pd.merge(left = nhl_df_local_2018_grouped, right = population, left_index = True, right_index = True)
out_df_nhl.drop(columns = ['win', 'loss'], inplace = True)


def nhl_correlation(): 
    win_loss_by_region =nhl_df_local_2018_grouped['win_loss_ratio'].values 
    win_loss_by_region
    print(len(population_by_region))
    print(len(win_loss_by_region))
    print(cities_local['Metropolitan area'])
    print(nhl_df_local_2018_grouped['win_loss_ratio'].index.values)

    assert len(population_by_region) == len(win_loss_by_region), "Q1: Your lists must be the same length"
    assert len(population_by_region) == 28, "Q1: There should be 28 teams being analysed for NHL"
    #return 1
    corr, _ = stats.pearsonr(population_by_region, win_loss_by_region)
    print(corr)
    return corr
    raise NotImplementedError() 

def get_nhl_data():
    return out_df_nhl    

C:\Users\singl\AppData\Local\Temp\ipykernel_104588\3092091601.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cities_local.sort_values(by = 'Metropolitan area', inplace = True)
C:\Users\singl\AppData\Local\Temp\ipykernel_104588\3092091601.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cities_local.drop(labels = ['—',  ''], axis = 0, inplace = True)


,team,GP,W,L,OL,PTS,PTS%,GF,GA,SRS,SOS,RPt%,ROW,year,League,area
1,Tampa Bay Lightning,82,54,23,5,113,.689,296,236,0.66,-0.07,.634,48,2018,NHL,Tampa Bay Area
2,Boston Bruins,82,50,20,12,112,.683,270,214,0.62,-0.07,.610,47,2018,NHL,Boston
3,Toronto Maple Leafs,82,49,26,7,105,.640,277,232,0.49,-0.06,.567,42,2018,NHL,Toronto
4,Florida Panthers,82,44,30,8,96,.585,248,246,-0.01,-0.04,.537,41,2018,NHL,Miami–Fort Lauderdale
5,Detroit Red Wings,82,30,39,13,73,.445,217,255,-0.48,-0.01,.341,25,2018,NHL,Detroit
6,Montreal Canadiens,82,29,40,13,71,.433,209,264,-0.68,0.00,.378,27,2018,NHL,Montreal
7,Ottawa Senators,82,28,43,11,67,.409,221,291,-0.85,0.00,.372,26,2018,NHL,Ottawa
8,Buffalo Sabres,82,25,45,12,62,.378,199,280,-0.98,0.01,.311,24,2018,NHL,Buffalo
10,Washington Capitals,82,49,26,7,105,.640,259,239,0.21,-0.04,.585,46,2018,NHL,"Washington, D.C."
11,Pittsburgh Penguins,82,47,29,6,100,.610,272,250,0.23,-0.04,.573,45,2018,NHL,Pittsburgh


,win,loss,win_loss_ratio
area,,,
Boston,50,20,0.714286
Buffalo,25,45,0.357143
Calgary,37,35,0.513889
Chicago,33,39,0.458333
Columbus,45,30,0.600000
Dallas–Fort Worth,42,32,0.567568
Denver,43,30,0.589041
Detroit,30,39,0.434783
Edmonton,36,40,0.473684


In [2]:
nhl_correlation()

28
28
NHL
Bruins                                      Boston
Sabres                                     Buffalo
Flames                                     Calgary
Blackhawks                                 Chicago
Blue Jackets                              Columbus
Stars                            Dallas–Fort Worth
Avalanche                                   Denver
Red Wings                                  Detroit
Oilers                                    Edmonton
Golden Knights                           Las Vegas
Kings Ducks                            Los Angeles
Panthers                     Miami–Fort Lauderdale
Wild                        Minneapolis–Saint Paul
Canadiens                                 Montreal
Predators                                Nashville
Rangers Islanders Devils             New York City
Senators                                    Ottawa
Flyers                                Philadelphia
Coyotes                                    Phoenix
Penguins             

0.012308996455744264

## Question 2
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NBA** using **2018** data.

In [3]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

nba_df=pd.read_csv("assets/nba.csv")
cities=pd.read_html("assets/wikipedia_data.html")[1]
cities=cities.iloc[:-1,[0,3,5,6,7,8]]

nhl_df_local=nba_df
cities_local=cities

pattern = r'\[note\s?[0-9]*\]'
for x in cities_local.columns: 
    cities_local[x] = cities_local[x].str.replace(pat = pattern,repl = '', case=False, flags = re.IGNORECASE, regex = True)

pattern = r'[*+]'
for x in cities_local.columns: 
    cities_local[x] = cities_local[x].str.replace(pat = pattern,repl = '', case=False, flags = re.IGNORECASE, regex = True)

cities_local.rename({"Population (2016 est.)[8]": "Population"}, axis =1, inplace = True)
#cities_local.columns = [ x for x in cities_local.columns]
population_by_region = [] # pass in metropolitan area population from cities_local
win_loss_by_region = [] # pass in win/loss ratio from nhl_df_local in the same order as cities_local["Metropolitan area"]

cities_local = cities_local[['Metropolitan area', 'NBA', 'Population']]
citites = cities_local.set_index('NBA', inplace = True)
cities_local.sort_values(by = 'Metropolitan area', inplace = True)
cities_local.drop(labels = ['—',  ''], axis = 0, inplace = True)
cities_local.head(3)
population = cities_local[['Population', 'Metropolitan area']]
population.set_index('Metropolitan area', inplace = True)
population.head(3)
population_by_region = cities_local['Population'].values.astype(int)

nhl_df_local_2018 = nhl_df_local[nhl_df_local['year'] ==2018]
teams_to_drop = ['Atlantic Division', 'Northwest Division', 'Southwest Division', 'Southeast Division', 'Northeast Division','Central Division', 'Pacific Division'] 

nhl_df_local_2018 = nhl_df_local_2018[~nhl_df_local_2018['team'].isin(teams_to_drop)] 

nhl_df_local_2018.head()

pattern = r'\*?.\([0-9]*\)'
for x in nhl_df_local_2018.columns: 
    nhl_df_local_2018[x]=nhl_df_local_2018[x].astype('str')
    nhl_df_local_2018[x] = nhl_df_local_2018[x].str.replace(pat = pattern,repl = '', case=False, flags = re.IGNORECASE, regex = True)

nhl_df_local_2018['area'] = nhl_df_local_2018['team'].apply(lambda x: x.split(' ')[-1])
#print(nhl_df_local_2018.head(5))
#print(cities_local.index.values)
#print(nhl_df_local_2018[['team','area']])

def lookup_area_by_team(nfl_team):
    for team in cities_local.index.values:
        if nfl_team in team:
            return cities_local.at[team,'Metropolitan area']

nhl_df_local_2018['area'] = nhl_df_local_2018['area'].apply(lookup_area_by_team)


#print(nhl_df_local_2018['team'].apply(lambda x: x.split(' ')[-1])

nhl_df_local_2018['W']=nhl_df_local_2018['W'].astype('int')
nhl_df_local_2018['L']=nhl_df_local_2018['L'].astype('int')
nhl_df_local_2018['win_loss_ratio'] = nhl_df_local_2018['W']/ (nhl_df_local_2018['W'] + nhl_df_local_2018['L'])
nhl_df_local_2018_grouped = nhl_df_local_2018.groupby(by= ['area']).agg(win = ('W','sum') , loss=( 'L', 'sum'), win_loss_ratio = ('win_loss_ratio', 'mean'))
nhl_df_local_2018_grouped 

#print(population)
#print(nhl_df_local_2018_grouped)

out_df_nba = pd.merge(left = nhl_df_local_2018_grouped, right = population, left_index = True, right_index = True)
out_df_nba.drop(columns = ['win', 'loss'], inplace = True)
print(out_df_nba)

def nba_correlation():

    win_loss_by_region =nhl_df_local_2018_grouped['win_loss_ratio'].values 
    win_loss_by_region
    #print(len(population_by_region))
    #print(len(win_loss_by_region))
    #print(population_by_region)
    #print(win_loss_by_region)

    assert len(population_by_region) == len(win_loss_by_region), "Q2: Your lists must be the same length"
    assert len(population_by_region) == 28, "Q2: There should be 28 teams being analysed for NBA"
    corr, _ = stats.pearsonr(win_loss_by_region,population_by_region)
    print(corr)
    return corr
    raise NotImplementedError() 

def get_nba_data():
    return out_df_nba

                        win_loss_ratio Population
area                                             
Atlanta                       0.292683    5789700
Boston                        0.670732    4794447
Charlotte                     0.439024    2474314
Chicago                       0.329268    9512999
Cleveland                     0.609756    2055612
Dallas–Fort Worth             0.292683    7233323
Denver                        0.560976    2853077
Detroit                       0.475610    4297617
Houston                       0.792683    6772470
Indianapolis                  0.585366    2004230
Los Angeles                   0.469512   13310447
Memphis                       0.268293    1342842
Miami–Fort Lauderdale         0.536585    6066387
Milwaukee                     0.536585    1572482
Minneapolis–Saint Paul        0.573171    3551036
New Orleans                   0.585366    1268883
New York City                 0.347561   20153634
Oklahoma City                 0.585366    1373211


C:\Users\singl\AppData\Local\Temp\ipykernel_104588\1320899563.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cities_local.sort_values(by = 'Metropolitan area', inplace = True)
C:\Users\singl\AppData\Local\Temp\ipykernel_104588\1320899563.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cities_local.drop(labels = ['—',  ''], axis = 0, inplace = True)


In [4]:
nba_correlation()

-0.17657160252844611


-0.17657160252844611

## Question 3
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **MLB** using **2018** data.

In [5]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

mlb_df=pd.read_csv("assets/mlb.csv")
cities=pd.read_html("assets/wikipedia_data.html")[1]
cities=cities.iloc[:-1,[0,3,5,6,7,8]]

nhl_df_local=mlb_df
cities_local=cities

pattern = r'\[note\s?[0-9]*\]'
for x in cities_local.columns: 
    cities_local[x] = cities_local[x].str.replace(pat = pattern,repl = '', case=False, flags = re.IGNORECASE, regex = True)

pattern = r'[*+]'
for x in cities_local.columns: 
    cities_local[x] = cities_local[x].str.replace(pat = pattern,repl = '', case=False, flags = re.IGNORECASE, regex = True)

cities_local.rename({"Population (2016 est.)[8]": "Population"}, axis =1, inplace = True)
#cities_local.columns = [ x for x in cities_local.columns]
population_by_region = [] # pass in metropolitan area population from cities_local
win_loss_by_region = [] # pass in win/loss ratio from nhl_df_local in the same order as cities_local["Metropolitan area"]

cities_local = cities_local[['Metropolitan area', 'MLB', 'Population']]
citites = cities_local.set_index('MLB', inplace = True)
cities_local.sort_values(by = 'Metropolitan area', inplace = True)
cities_local.drop(labels = ['—',  ''], axis = 0, inplace = True)
cities_local.head(3)
population = cities_local[['Population', 'Metropolitan area']]
population.set_index('Metropolitan area', inplace = True)
population.head(3)
population_by_region = cities_local['Population'].values.astype(int)

nhl_df_local_2018 = nhl_df_local[nhl_df_local['year'] ==2018]
teams_to_drop = ['Atlantic Division', 'Northwest Division', 'Southwest Division', 'Southeast Division', 'Northeast Division','Central Division', 'Pacific Division'] 

nhl_df_local_2018 = nhl_df_local_2018[~nhl_df_local_2018['team'].isin(teams_to_drop)] 

nhl_df_local_2018.head()

pattern = r'\*?.\([0-9]*\)'
for x in nhl_df_local_2018.columns: 
    nhl_df_local_2018[x]=nhl_df_local_2018[x].astype('str')
    nhl_df_local_2018[x] = nhl_df_local_2018[x].str.replace(pat = pattern,repl = '', case=False, flags = re.IGNORECASE, regex = True)

nhl_df_local_2018['area'] = nhl_df_local_2018['team'].apply(lambda x: x.split(' ')[-1])
#print(nhl_df_local_2018.head(5))
#print(cities_local.index.values)
#print(nhl_df_local_2018[['team','area']])

def lookup_area_by_team(nfl_team):
    for team in cities_local.index.values:
        if nfl_team in team:
            return cities_local.at[team,'Metropolitan area']

nhl_df_local_2018['area'] = nhl_df_local_2018['area'].apply(lookup_area_by_team)

from IPython.display import HTML

display(HTML(nhl_df_local_2018.to_html()))

nhl_df_local_2018['area'] = nhl_df_local_2018['area'].where(~(nhl_df_local_2018['team']=='Chicago White Sox'), 'Chicago')
display(HTML(nhl_df_local_2018.to_html()))

#print(nhl_df_local_2018['team'].apply(lambda x: x.split(' ')[-1])

nhl_df_local_2018['W']=nhl_df_local_2018['W'].astype('int')
nhl_df_local_2018['L']=nhl_df_local_2018['L'].astype('int')
nhl_df_local_2018['win_loss_ratio'] = nhl_df_local_2018['W']/ (nhl_df_local_2018['W'] + nhl_df_local_2018['L'])
nhl_df_local_2018['win_loss_ratio'] = nhl_df_local_2018['W-L%']
nhl_df_local_2018['win_loss_ratio'] = nhl_df_local_2018['win_loss_ratio'].astype('float')
nhl_df_local_2018_grouped = nhl_df_local_2018.groupby(by= ['area']).agg(win = ('W','sum') , loss=( 'L', 'sum'), win_loss_ratio = ('win_loss_ratio', 'mean'))
nhl_df_local_2018_grouped  
#print(population)
#print(nhl_df_local_2018_grouped)

out_df_mlb = pd.merge(left = nhl_df_local_2018_grouped, right = population, left_index = True, right_index = True)
out_df_mlb.drop(columns = ['win', 'loss'], inplace = True)
print(out_df_mlb)

def mlb_correlation(): 
    print(len(nhl_df_local_2018_grouped))
    print(nhl_df_local_2018_grouped)
    win_loss_by_region =nhl_df_local_2018_grouped['win_loss_ratio'].values 
    win_loss_by_region
    print(len(population_by_region))
    print(len(win_loss_by_region))
    print(population_by_region)
    print(win_loss_by_region)
    
    assert len(population_by_region) == len(win_loss_by_region), "Q3: Your lists must be the same length"
    assert len(population_by_region) == 26, "Q3: There should be 26 teams being analysed for MLB"
    
    return stats.pearsonr(population_by_region, win_loss_by_region)
    raise NotImplementedError()

def get_mlb_data():
    return out_df_mlb

C:\Users\singl\AppData\Local\Temp\ipykernel_104588\3262131381.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cities_local.sort_values(by = 'Metropolitan area', inplace = True)
C:\Users\singl\AppData\Local\Temp\ipykernel_104588\3262131381.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cities_local.drop(labels = ['—',  ''], axis = 0, inplace = True)


,team,W,L,W-L%,GB,year,League,area
0,Boston Red Sox,108,54,0.667,--,2018,MLB,Boston
1,New York Yankees,100,62,0.617,8.0,2018,MLB,New York City
2,Tampa Bay Rays,90,72,0.556,18.0,2018,MLB,Tampa Bay Area
3,Toronto Blue Jays,73,89,0.451,35.0,2018,MLB,Toronto
4,Baltimore Orioles,47,115,0.29,61.0,2018,MLB,Baltimore
5,Cleveland Indians,91,71,0.562,--,2018,MLB,Cleveland
6,Minnesota Twins,78,84,0.481,13.0,2018,MLB,Minneapolis–Saint Paul
7,Detroit Tigers,64,98,0.395,27.0,2018,MLB,Detroit
8,Chicago White Sox,62,100,0.3829999999999999,29.0,2018,MLB,Boston
9,Kansas City Royals,58,104,0.358,33.0,2018,MLB,Kansas City


,team,W,L,W-L%,GB,year,League,area
0,Boston Red Sox,108,54,0.667,--,2018,MLB,Boston
1,New York Yankees,100,62,0.617,8.0,2018,MLB,New York City
2,Tampa Bay Rays,90,72,0.556,18.0,2018,MLB,Tampa Bay Area
3,Toronto Blue Jays,73,89,0.451,35.0,2018,MLB,Toronto
4,Baltimore Orioles,47,115,0.29,61.0,2018,MLB,Baltimore
5,Cleveland Indians,91,71,0.562,--,2018,MLB,Cleveland
6,Minnesota Twins,78,84,0.481,13.0,2018,MLB,Minneapolis–Saint Paul
7,Detroit Tigers,64,98,0.395,27.0,2018,MLB,Detroit
8,Chicago White Sox,62,100,0.3829999999999999,29.0,2018,MLB,Chicago
9,Kansas City Royals,58,104,0.358,33.0,2018,MLB,Kansas City


                        win_loss_ratio Population
area                                             
Atlanta                          0.556    5789700
Baltimore                        0.290    2798886
Boston                           0.667    4794447
Chicago                          0.483    9512999
Cincinnati                       0.414    2165139
Cleveland                        0.562    2055612
Dallas–Fort Worth                0.414    7233323
Denver                           0.558    2853077
Detroit                          0.395    4297617
Houston                          0.636    6772470
Kansas City                      0.358    2104509
Los Angeles                      0.529   13310447
Miami–Fort Lauderdale            0.391    6066387
Milwaukee                        0.589    1572482
Minneapolis–Saint Paul           0.481    3551036
New York City                    0.546   20153634
Philadelphia                     0.494    6070500
Phoenix                          0.506    4661537


In [6]:
mlb_correlation()

26
                        win  loss  win_loss_ratio
area                                             
Atlanta                  90    72           0.556
Baltimore                47   115           0.290
Boston                  108    54           0.667
Chicago                 157   168           0.483
Cincinnati               67    95           0.414
Cleveland                91    71           0.562
Dallas–Fort Worth        67    95           0.414
Denver                   91    72           0.558
Detroit                  64    98           0.395
Houston                 103    59           0.636
Kansas City              58   104           0.358
Los Angeles             172   153           0.529
Miami–Fort Lauderdale    63    98           0.391
Milwaukee                96    67           0.589
Minneapolis–Saint Paul   78    84           0.481
New York City           177   147           0.546
Philadelphia             80    82           0.494
Phoenix                  82    80           0.5

PearsonRResult(statistic=0.15003737475409495, pvalue=0.46442827201123316)

## Question 4
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NFL** using **2018** data.

In [7]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re
import scipy.stats as stats

nfl_df=pd.read_csv("assets/nfl.csv")
cities=pd.read_html("assets/wikipedia_data.html")[1]
cities=cities.iloc[:-1,[0,3,5,6,7,8]]

nhl_df_local=nfl_df
cities_local=cities

pattern = r'\[note\s?[0-9]*\]'
for x in cities_local.columns: 
    cities_local[x] = cities_local[x].str.replace(pat = pattern,repl = '', case=False, flags = re.IGNORECASE, regex = True)

pattern = r'[*+]'
for x in cities_local.columns: 
    cities_local[x] = cities_local[x].str.replace(pat = pattern,repl = '', case=False, flags = re.IGNORECASE, regex = True)

cities_local.rename({"Population (2016 est.)[8]": "Population"}, axis =1, inplace = True)
#cities_local.columns = [ x for x in cities_local.columns]
population_by_region = [] # pass in metropolitan area population from cities_local
win_loss_by_region = [] # pass in win/loss ratio from nhl_df_local in the same order as cities_local["Metropolitan area"]

cities_local = cities_local[['Metropolitan area', 'NFL', 'Population']]
citites = cities_local.set_index('NFL', inplace = True)
cities_local.drop(labels = ['—', ''], axis = 0, inplace = True)
cities_local.drop(cities_local[cities_local['Metropolitan area'] =='Toronto'].index, inplace = True)

#print(cities_local)

population = cities_local[['Population', 'Metropolitan area']]
population.set_index('Metropolitan area', inplace = True)
population.head(3)
population_by_region = cities_local['Population'].values.astype(int)

nhl_df_local_2018 = nhl_df_local[nhl_df_local['year'] ==2018]
teams_to_drop = ['AFC East', 'AFC North', 'AFC South', 'AFC West', 'NFC East', 'NFC North', 'NFC South', 'NFC West'] 

nhl_df_local_2018 = nhl_df_local_2018[~nhl_df_local_2018['team'].isin(teams_to_drop)] 

#print(nhl_df_local_2018['team'])

pattern = r'[\*+]?'
for x in nhl_df_local_2018.columns: 
    nhl_df_local_2018[x]=nhl_df_local_2018[x].astype('str')
    nhl_df_local_2018[x] = nhl_df_local_2018[x].str.replace(pat = pattern,repl = '', case=False, flags = re.IGNORECASE, regex = True)

nhl_df_local_2018['area'] = nhl_df_local_2018['team'].apply(lambda x: x.split(' ')[-1])
#print(nhl_df_local_2018.head(5))
#print(cities_local.index.values)
#print(nhl_df_local_2018[['team','area']])

def lookup_area_by_team(nfl_team):
    for team in cities_local.index.values:
        if nfl_team in team:
            return cities_local.at[team,'Metropolitan area']

nhl_df_local_2018['area'] = nhl_df_local_2018['area'].apply(lookup_area_by_team)
nhl_df_local_2018['area'] = nhl_df_local_2018['area'].where(~(nhl_df_local_2018['team']=='Boston Red Sox'), 'Boston')
#print(len(nhl_df_local_2018['area']))

#print(nhl_df_local_2018['team'].apply(lambda x: x.split(' ')[-1])

nhl_df_local_2018_grouped = nhl_df_local_2018.groupby(by= ['area']).agg(win = ('W','sum') , loss=( 'L', 'sum'))
nhl_df_local_2018_grouped['win']=nhl_df_local_2018_grouped['win'].astype('int')
nhl_df_local_2018_grouped['loss']=nhl_df_local_2018_grouped['loss'].astype('int')
nhl_df_local_2018_grouped['win_loss_ratio'] =nhl_df_local_2018_grouped['win']/(nhl_df_local_2018_grouped['win']+ nhl_df_local_2018_grouped['loss'])

#print(population)
#print(nhl_df_local_2018_grouped)

out_df_nfl = pd.merge(left = nhl_df_local_2018_grouped, right = population, left_index = True, right_index = True)
out_df_nfl.drop(columns = ['win', 'loss'], inplace = True)
print(out_df_nfl)

def nfl_correlation(): 
    # YOUR CODE HERE
    #print(len(nhl_df_local_2018_grouped))
    #print(nhl_df_local_2018_grouped)
    
    win_loss_by_region =nhl_df_local_2018_grouped['win_loss_ratio'].values 
    win_loss_by_region
    
    print(len(population_by_region))
    print(len(win_loss_by_region))
    
    print(population_by_region)
    print(win_loss_by_region)

    assert len(population_by_region) == len(win_loss_by_region), "Q4: Your lists must be the same length"
    assert len(population_by_region) == 29, "Q4: There should be 29 teams being analysed for NFL"

    return stats.pearsonr(population_by_region, win_loss_by_region)
    raise NotImplementedError()

def get_nfl_data():
    return out_df_nfl

C:\Users\singl\AppData\Local\Temp\ipykernel_104588\1798750995.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cities_local.drop(labels = ['—', ''], axis = 0, inplace = True)
C:\Users\singl\AppData\Local\Temp\ipykernel_104588\1798750995.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cities_local.drop(cities_local[cities_local['Metropolitan area'] =='Toronto'].index, inplace = True)


                        win_loss_ratio Population
Atlanta                       0.437500    5789700
Baltimore                     0.625000    2798886
Boston                        0.687500    4794447
Buffalo                       0.375000    1132804
Charlotte                     0.437500    2474314
Chicago                       0.750000    9512999
Cincinnati                    0.375000    2165139
Cleveland                     0.466667    2055612
Dallas–Fort Worth             0.625000    7233323
Denver                        0.375000    2853077
Detroit                       0.375000    4297617
Green Bay                     0.400000     318236
Houston                       0.687500    6772470
Indianapolis                  0.625000    2004230
Jacksonville                  0.312500    1478212
Kansas City                   0.750000    2104509
Los Angeles                   0.965764   13310447
Miami–Fort Lauderdale         0.437500    6066387
Minneapolis–Saint Paul        0.533333    3551036


In [8]:
nfl_correlation()

29
29
[20153634 13310447  6657982  9512999  7233323  6131977  6070500  4794447
  3551036  2853077  6066387  4661537  4297617  6772470  5789700  3032171
  2342299  2055612  3798902  2165139  2104509  2798886  2474314  2004230
  1865298  1268883  1132804  1478212   318236]
[0.4375     0.625      0.6875     0.375      0.4375     0.75
 0.375      0.46666667 0.625      0.375      0.375      0.4
 0.6875     0.625      0.3125     0.75       0.96576433 0.4375
 0.53333333 0.5625     0.8125     0.03582803 0.5625     0.1875
 0.6        0.03503185 0.625      0.3125     0.4375    ]


PearsonRResult(statistic=0.03190154732390762, pvalue=0.8695108745624701)

## Question 5
In this question I would like you to explore the hypothesis that **given that an area has two sports teams in different sports, those teams will perform the same within their respective sports**. How I would like to see this explored is with a series of paired t-tests (so use [`ttest_rel`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html)) between all pairs of sports. Are there any sports where we can reject the null hypothesis? Again, average values where a sport has multiple teams in one region. Remember, you will only be including, for each sport, cities which have teams engaged in that sport, drop others as appropriate. This question is worth 20% of the grade for this assignment.

In [9]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

mlb_df=pd.read_csv("assets/mlb.csv")
nhl_df=pd.read_csv("assets/nhl.csv")
nba_df=pd.read_csv("assets/nba.csv")
nfl_df=pd.read_csv("assets/nfl.csv")
cities=pd.read_html("assets/wikipedia_data.html")[1]

MLB = get_mlb_data().drop('Population', axis=1)
NHL = get_nhl_data().drop('Population', axis=1)
NBA = get_nba_data().drop('Population', axis=1)
NFL = get_nfl_data().drop('Population', axis=1)


cities=cities.iloc[:-1,[0,3,5,6,7,8]]

data_set = {'NFL': NFL,
            'NBA': NBA,
            'NHL': NHL,
            'MLB': MLB}
sports = ['NFL', 'NBA', 'NHL', 'MLB']

def get_p_value(k):
    p_values = []
    for each in sports:
        print(k)
        print(data_set[k])
        print(each)
        print(data_set[each])
        
        df = pd.merge(data_set[k], data_set[each], how="outer", left_index=True, right_index=True)
        df.dropna(how = 'all', inplace = True)
        #print(df)
        corr = stats.ttest_rel(df['win_loss_ratio_x'], df['win_loss_ratio_y'], nan_policy ='omit')[1]
        #print("\n Correlation", corr)
        nhl_corr = round(corr, 2)
        
        p_values.append(corr)
    print(p_values)
    return p_values

def sports_team_performance():
    # YOUR CODE HERE

    # Note: p_values is a full dataframe, so df.loc["NFL","NBA"] should be the same as df.loc["NBA","NFL"] and
    # df.loc["NFL","NFL"] should return np.nan
    sports = ['NFL', 'NBA', 'NHL', 'MLB']
    p_values = pd.DataFrame({k:get_p_value(k) for k in sports}, index=sports)
    print(p_values)
    # assert abs(p_values.loc["NBA", "NHL"] - 0.02) <= 1e-2, "The NBA-NHL p-value should be around 0.02"
    # assert abs(p_values.loc["MLB", "NFL"] - 0.80) <= 0.10, "The MLB-NFL p-value should be around 0.80"
    return p_values
    raise NotImplementedError()

In [10]:
sports_team_performance()











































































NFL
                        win_loss_ratio
Atlanta                       0.437500
Baltimore                     0.625000
Boston                        0.687500
Buffalo                       0.375000
Charlotte                     0.437500
Chicago                       0.750000
Cincinnati                    0.375000
Cleveland                     0.466667
Dallas–Fort Worth             0.625000
Denver                        0.375000
Detroit                       0.375000
Green Bay                     0.400000
Houston                       0.687500
Indianapolis                  0.625000
Jacksonville                  0.312500
Kansas City                   0.750000
Los Angeles                   0.965764
Miami–Fort Lauderdale         0.437500
Minneapolis–Saint Paul        0.533333
Nashville                     0.562500
New Orleans                   0.812500
New York City                 0.035828
Philadelphia                  0.562500
Phoenix                       0.187500
Pittsburgh           

,NFL,NBA,NHL,MLB
NFL,NaN,0.858349,0.059229,0.974308
NBA,0.858349,NaN,0.022316,0.951629
NHL,0.059229,0.022316,NaN,0.000703
MLB,0.974308,0.951629,0.000703,NaN


In [11]:
def result():
    s = 'ACAABAACAAABACDBADDDFSDDDFFSSSASDAFAAACBAAAFASD'

    result = []
    # compete the pattern below
    pattern = r'([^A])AAA'
    for item in re.finditer(pattern, s):
      # identify the group number below.
      result.append(item.group())
      
    return result

In [12]:
result()

['CAAA', 'FAAA', 'BAAA']

In [13]:
my_list = [4,7,-5,3]
index = ['d', 'b', 'a', 'c']

df = pd.DataFrame(data = my_list, index = index, columns = ['col'])
df.iloc[0]

col    4
Name: d, dtype: int64

In [14]:
import re
s = 'ABCAC'
re.split('A',s)

['', 'BC', 'C']

In [15]:
re.search('A',s)

<re.Match object; span=(0, 1), match='A'>

In [16]:
bool(re.match('A',s)) == True

True

In [17]:
import numpy as np

a = np.arange(8)
print(a[4])
b = a[4:6]
b[:] = 40
c = a[4] + a[6]
c

4


46

In [18]:
def result():
    s = 'ACAABAACAAABACDBADDDFSDDDFFSSSASDAFAAACBAAAFASD'

    result = []
    # compete the pattern below
    pattern = r'([^A])(?=AAA)'
    for item in re.finditer(pattern, s):
      # identify the group number below.
      result.append(item.group())
      
    return result

In [19]:
result()

['C', 'F', 'B']